# Day 043 · yfinance 拉数据 · 中国版
**yfinance** · 阶段 P2 · Python 量化工具栈

> 前面几节用的都是国内的 baostock。这一节把视野放到全球:yfinance 是一个免费、不用注册、不用 token 的小工具,一行就能把美股(还有港股、外汇、加密、指数)的开高低收成交量拉回来。我们用它批量拉英伟达、特斯拉、微软、亚马逊、Meta 这五只人人都听过的美股,比一比5 年谁涨得最猛,顺便讲清两个新手必踩的坑:复权不做会被拆股骗出一个假暴跌、批量拉数据不限速会被网站封。

---

### 关于「中国版」

本 notebook 是为**国内学员**优化的版本:
- 数据源用 **akshare**(国内可访问、零 VPN、免注册),取代了视频里的 yfinance
- 标的尽量保持原意:美股 ETF→A 股 ETF / 国际公司→A 股龙头
- 所讲的**概念和方法 100% 一致**,但**具体数字可能与视频里略有差异**(因为是不同时间窗 / 不同标的)
- 一般情况国内 `pip install akshare` 即可,无需 token / VPN

**课件生成日期:** 2026-06-14  ·  **建议学习时长:** 16 分钟

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有必需的 Python 包(含 `akshare`),缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续


In [5]:
# === 环境自检 + 自动安装(运行此单元格即可)===
import importlib, subprocess, sys, os

REQUIRED = ["akshare", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels"]
PIP_NAME = {"sklearn":"scikit-learn","cv2":"opencv-python","PIL":"Pillow","bs4":"beautifulsoup4","yaml":"PyYAML"}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))
if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置 ===
import matplotlib, matplotlib.pyplot as plt, matplotlib.font_manager as fm
CJK = ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
       "C:/Windows/Fonts/msyh.ttc","C:/Windows/Fonts/simhei.ttf",
       "/System/Library/Fonts/PingFang.ttc","/System/Library/Fonts/STHeiti Medium.ttc"]
for p in CJK:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP","Microsoft YaHei","PingFang SC","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪")


✓ 所有 8 个必需库已就绪
✓ 中文字体已加载:NotoSansCJK-Regular.ttc
✓ 环境就绪


## 🔌 第二步:加载国内数据助手

下面这一格是**工具函数**(可以折叠,不需要修改)。它把 `yfinance` 风格的 ticker(如 `600519.SS`)自动路由到对应的 akshare 接口,提供 `get_close(ticker)` 和 `get_close_multi(tickers_dict)` 两个函数。

In [6]:
# === 国内数据源助手(akshare 后端,不需要 VPN)===
# 这一格是工具函数,可以折叠,不需要修改。
# 它把 yfinance 风格的 ticker(如 "600519.SS" / "0700.HK" / "AAPL" / "BTC-USD")
# 自动路由到对应的 akshare 接口,统一返回 yfinance 风格的 Close DataFrame。

import re
from datetime import datetime, timedelta
import pandas as pd
import akshare as ak

_TICKER_MAP = {
    "^GSPC": ("us_index_sina", ".INX"),
    "^DJI":  ("us_index_sina", ".DJI"),
    "^IXIC": ("us_index_sina", ".IXIC"),
    "GC=F":  ("foreign_futures", "GC"),
    "SI=F":  ("foreign_futures", "SI"),
    "CL=F":  ("foreign_futures", "CL"),
    "BTC-USD": ("crypto", "BTC"),
    "ETH-USD": ("crypto", "ETH"),
}

def _retry(fn, *args, _retries=4, _wait=1.5, **kwargs):
    """akshare 上游(东方财富/新浪/Binance)偶有 RemoteDisconnected / Timeout,自动重试 4 次。
    2026-05-11 加:用户跑 cn notebook 拉 002594.SZ 时上游断连 → 整节卡死。
    每次重试间隔 _wait 秒(指数退避 1x → 1.5x → 2.25x)。
    """
    import time as _t
    last_err = None
    wait = _wait
    for i in range(_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            name = type(e).__name__
            if i == _retries - 1:
                print(f"  ✗ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次仍失败({name}),放弃")
                raise
            print(f"  ⚠ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次失败({name}),{wait:.1f}s 后重试")
            _t.sleep(wait)
            wait *= 1.5

def _parse_period(period):
    end = datetime.today()
    m = re.match(r"^(\d+)\s*(y|mo|d|w)$", period.lower().strip())
    days = 365 * 3 if not m else int(m.group(1)) * {"y":365,"mo":30,"w":7,"d":1}[m.group(2)]
    return (end - timedelta(days=days+30)).strftime("%Y%m%d"), end.strftime("%Y%m%d")

def _classify(ticker):
    t = ticker.strip()
    if t in _TICKER_MAP: return _TICKER_MAP[t]
    if t.endswith((".SS",".SH",".SZ")):
        code = t.split(".")[0]
        if code.startswith(("51","159","58")) or code in ("510300","510500","510050","511010","513100"):
            return ("a_etf", code)
        if code in ("000300","000016","000905","000852","000001"):
            return ("a_index", code)
        return ("a_stock", code)
    if t.endswith(".HK"):
        return ("hk", t.split(".")[0].zfill(5))
    return ("us", t)

def _norm(df, dc, cc):
    out = df[[dc, cc]].copy()
    out[dc] = pd.to_datetime(out[dc])
    return out.set_index(dc).sort_index()[cc].astype(float).rename("Close")

def get_close(ticker, period="3y"):
    """返回某标的 Close 价格 series。后端 akshare,中国可访问。
    所有 ak.* 调用都过 _retry(4 次,指数退避)— 防东方财富/新浪上游瞬时断连。
    """
    start, end = _parse_period(period)
    kind, sym = _classify(ticker)
    if kind == "a_stock":
        return _norm(_retry(ak.stock_zh_a_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_etf":
        return _norm(_retry(ak.fund_etf_hist_em, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_index":
        idx_map = {"000300":"sh000300","000016":"sh000016","000905":"sh000905","000852":"sh000852","000001":"sh000001"}
        s = _norm(_retry(ak.stock_zh_index_daily_em, symbol=idx_map.get(sym, f"sh{sym}")), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "hk":
        return _norm(_retry(ak.stock_hk_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "us":
        s = _norm(_retry(ak.stock_us_daily, symbol=sym, adjust="qfq"), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "us_index_sina":
        s = _norm(_retry(ak.index_us_stock_sina, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "foreign_futures":
        s = _norm(_retry(ak.futures_foreign_hist, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "crypto":
        import requests as _rq
        def _binance():
            r = _rq.get("https://api.binance.com/api/v3/klines",
                        params={"symbol": f"{sym}USDT", "interval": "1d", "limit": 1000}, timeout=15)
            r.raise_for_status()
            return r.json()
        klines = _retry(_binance)
        df = pd.DataFrame(klines, columns=["open_time","open","high","low","close","volume",
                                            "close_time","qav","trades","tbb","tbq","ignore"])
        df["date"] = pd.to_datetime(df["open_time"], unit="ms")
        df["close"] = df["close"].astype(float)
        s = df.set_index("date").sort_index()["close"].rename("Close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    raise ValueError(f"unsupported ticker: {ticker}")

def get_close_multi(tickers, period="3y"):
    """批量取 Close,返回 DataFrame,列名是 tickers dict 的 key(中文名),按交集日期对齐。"""
    series = {name: get_close(t, period=period) for name, t in tickers.items()}
    return pd.concat(series, axis=1).sort_index()

print("✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据")


✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据


## 学习目标

- 会用 yf.Ticker(代码).history(period='5y') 一行拿到一只股票多年的开高低收成交量
- 会用循环 + time.sleep 限速批量拉一篮子股票,并用 dict 构造合并表防止列错位
- 理解复权(auto_adjust)的重要性:不复权会把拆股当成暴跌,算收益全错
- 会用 actions 查一只股票的分红和拆股历史
- 会用 isna 校验缺失值,知道停牌、上市晚会留下空洞,要按市场对齐后再算

## 历史背景:老张手动下了五个 CSV,还把英伟达的“暴跌”当成了崩盘

老张想验证一个简单想法:这五只美国科技龙头,5 年前各买10000 块,今天各值多少?他老老实实跑去财经网站,一只一只点“导出 CSV”,下了五个文件,光是点鼠标就点到手酸。更糟的是,他下英伟达时随手选了“不复权”的版本,打开一看,2024 年某一天股价从 1000 多美元直接跌到 100 出头,他吓出一身冷汗,以为公司出了大事差点割肉——其实那天是英伟达一拆十,1 股变 10 股、股价自然除以 10,公司啥事没有。后来他学了 yfinance,一个循环就把五只股票5 年数据全拉回来,而且默认就是复权的连续价格,再没被假暴跌吓到。从手动点五个 CSV,到一段代码全自动,这就是数据工具帮你省下的时间和踩过的坑。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. yfinance:免费拉全球行情的小工具

yfinance 是一个开源 Python 包,背后连的是 Yahoo Finance 的公开数据,免费、不用注册、不用申请 key。一行 yf.Ticker('NVDA').history(period='5y') 就能拿到英伟达5 年的开高低收成交量。除了美股,港股(代码后缀 .HK)、A 股(.SS/.SZ)、外汇、加密货币、指数都能拉,是入门拿数据最省事的选择。


### 2. history 与 download 两种拉法

拉一只股票用 yf.Ticker('代码').history(period='5y'),返回干净的单层列(Open/High/Low/Close/Volume)。一次拉一篮子可以用 yf.download(['NVDA','TSLA'], period='5y'),但它返回的是双层列(指标 × 代码),新手容易取错列。本节用循环 + dict 构造的方式批量拉,列名和数据一一对应,稳妥不出错。period 可以写 '1y'、'5y'、'max',也可以用 start/end 指定起止日期。


### 3. 复权 auto_adjust:不做会被拆股骗

公司会分红、会拆股,这些会让原始股价跳一下,但你的真实收益并没变,复权就是把这些影响抹平。yfinance 这里有个细节要分清:它返回的 Close 默认已经做了拆股调整(所以你看不到拆股断崖,Yahoo 替你处理了),而 Adj Close 是在此基础上把分红也算进去的「真复权价」。设 auto_adjust=True 时,拿到的 Close 就等于这个真复权价。算收益、做回测一律用复权价。提醒:有些数据源(比如下一节 akshare 的不复权选项)给的是完全没调过的原始价,那才会看到拆股当天的断崖,所以拉数据务必显式选复权。


### 4. actions:分红和拆股记录

yf.Ticker('NVDA').actions 返回这只股票历史上的分红(Dividends)和拆股(Stock Splits)记录。拆股那一列的 10.0 就表示一拆十。知道这些事件发生在哪天,能帮你理解原始价格为什么会跳、复权为什么必要。


### 5. 批量限速 + 缺失校验

批量拉几十只股票时,别一口气连续请求,否则容易被网站当成爬虫限流甚至临时封。规矩做法是每拉一只 time.sleep(1) 歇一下。拉回来还要用 isna().sum() 检查缺失:有的股票上市晚、有的停过牌,会留下空洞。把不同股票拼成一张表后,常用 dropna 只保留大家都有数据的交易日再算(注意跨市场要分市场对齐,本节全是美股可统一处理)。


## 实操:yfinance — Ticker.history 一行拉行情 / 批量循环 + 限速 / auto_adjust 复权 / actions 分红拆股 / 缺失校验

下面这段代码用 akshare 抓数据,国内零 VPN 跑通。**直接 Run All** 看结果。

**依赖:** `pip install pandas numpy matplotlib akshare statsmodels scipy`

In [7]:
# day_043_yfinance_中国版.py — yfinance 拉美股(国内网络版:yfinance 优先,失败/已有缓存自动读本地 CSV)
# 国内访问 yahoo 偶尔被墙/超时。本版按铁律 55 + 62 写双路径:CSV 在就先读,没有就 yfinance 拉再存到仓库根 data/
# 三张图跟原版一一对应:① 五年净值对比 ② 微软复权 Close vs Adj Close ③ 英伟达近一年 K 线 + 成交量
# 预下载(可选):python tools/fetch_lesson_data.py 43
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
# 路径自适应(铁律62):无论 notebook 在仓库根还是 out/notebook/cn/ 都能找到 data/
def _data_path(_name):
    from pathlib import Path as _P
    _here = _P.cwd()
    for _b in [_here/'data', _here/'..'/'data', _here/'..'/'..'/'data', _here/'..'/'..'/'..'/'data']:
        if (_b/_name).exists():
            return str(_b/_name)
    (_here/'data').mkdir(parents=True, exist_ok=True)
    return str(_here/'data'/_name)

pd.set_option('display.width', 140)
plt.rcParams['axes.unicode_minus'] = False
TICKERS = {'英伟达': 'NVDA', '特斯拉': 'TSLA', '微软': 'MSFT', '亚马逊': 'AMZN', 'Meta': 'META'}


def load_or_fetch(csv_path, fetch_fn, cols=None):
    """通用双路径(铁律 62):CSV 在就读,不在就用 fetch_fn 拉再存。失败时再尝试读 CSV 兜底。"""
    csv = Path(csv_path)
    if csv.exists():
        df = pd.read_csv(csv, index_col=0, parse_dates=True)
        print(f'数据来源:本地缓存 {csv}')
        return df[cols] if cols else df
    try:
        df = fetch_fn()
        csv.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(csv)
        print(f'数据来源:yfinance 实时拉取,已缓存到 {csv}')
        return df[cols] if cols else df
    except Exception as e:
        print(f'yfinance 拉取失败({type(e).__name__}: {e}),尝试读本地 CSV 兜底')
        df = pd.read_csv(csv, index_col=0, parse_dates=True)
        return df[cols] if cols else df


def fetch_basket():
    """拉一篮子美股 5 年收盘,dict 构造防错位(铁律 39)。"""
    import yfinance as yf
    closes = {}
    for name, tk in TICKERS.items():
        h = yf.Ticker(tk).history(period='5y', auto_adjust=True)
        if h.empty:
            raise RuntimeError(f'{tk} 返回空,可能被限流')
        closes[name] = h['Close']
        time.sleep(1)
    df = pd.DataFrame(closes)
    df.index = df.index.tz_localize(None)
    return df


def fetch_msft():
    """微软 5 年,auto_adjust=False 才能同时拿 Close(只调拆股)和 Adj Close(含分红)。"""
    import yfinance as yf
    m = yf.Ticker('MSFT').history(period='5y', auto_adjust=False)
    if m.empty:
        raise RuntimeError('MSFT 返回空,可能被限流')
    m.index = m.index.tz_localize(None)
    return m


def fetch_nvda1y():
    """英伟达近一年,带成交量画 K 线。"""
    import yfinance as yf
    one = yf.Ticker('NVDA').history(period='1y', auto_adjust=True)
    if one.empty:
        raise RuntimeError('NVDA 返回空,可能被限流')
    one.index = one.index.tz_localize(None)
    return one


# ====================================================================
print('==== 1. 拉一篮子美股:CSV 优先,失败回退 yfinance(国内网络更稳)====')
close = load_or_fetch(_data_path('D043_yfinance_中国版.csv'), fetch_basket, cols=list(TICKERS.keys()))
print('各只股票缺失天数(NaN):'); print(close.isna().sum())
close = close.dropna()
print(f'对齐后剩 {len(close)} 个共同交易日  {close.index[0].date()} → {close.index[-1].date()}')

# ====================================================================
print('\n==== 2. 五年涨幅大比拼(净值=各自起点归 1)====')
nv = close / close.iloc[0]
final = nv.iloc[-1].sort_values(ascending=False)
print('五年净值(起点 1,数字=翻了几倍):')
for name, v in final.items():
    print(f'  {name:6} {v:6.2f} 倍')

fig, ax = plt.subplots(figsize=(13, 6))
for name in final.index:
    ax.plot(nv.index, nv[name], lw=1.4, label=f'{name} {final[name]:.1f}x')
ax.set_title('五只美股五年净值对比(各自起点归 1)'); ax.set_ylabel('净值(倍)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('chart_01.png', dpi=120); plt.close()
print('图1(五年净值对比)已保存')

# ====================================================================
print('\n==== 3. 复权到底复了什么:Close 已调拆股,Adj Close 再把分红也算进去 ====')
m = load_or_fetch(_data_path('D043_yfinance_中国版_msft.csv'), fetch_msft)
close_split = m['Close']        # Yahoo 默认已做拆股调整,但没把分红算进去
adj = m['Adj Close']            # 拆股 + 分红都调好的真复权价
gap0 = (close_split.iloc[0] / adj.iloc[0] - 1) * 100
fig, ax = plt.subplots(figsize=(13, 5.5))
ax.plot(close_split.index, close_split, color='#bf616a', lw=1.3, label='Close(只调拆股,没算分红)')
ax.plot(adj.index, adj, color='#5e81ac', lw=1.3, label='Adj Close(拆股 + 分红,真复权价)')
ax.set_title(f'微软:Close vs Adj Close(起点差 {gap0:.1f}% 就是 5 年累计分红的贡献)')
ax.set_ylabel('价格(美元)'); ax.legend()
plt.tight_layout(); plt.savefig('chart_02.png', dpi=120); plt.close()
print(f'图2(复权对比)已保存:起点 Adj Close 比 Close 低 {gap0:.1f}%,这就是分红被算进真实收益')
print("Adj Close 才是含分红的真复权价;下一节 akshare 拉 A 股记得显式选复权(adjust='qfq')")

# ====================================================================
print('\n==== 4. 单只 K 线 + 成交量(英伟达近一年)====')
one = load_or_fetch(_data_path('D043_yfinance_中国版_nvda1y.csv'), fetch_nvda1y)
fig, (axp, axv) = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                               gridspec_kw={'height_ratios': [3, 1]})
axp.plot(one.index, one['Close'], color='#4c566a', lw=1.2)
axp.plot(one.index, one['Close'].rolling(20).mean(), color='#d08770', lw=1.1, label='20 日均线')
axp.set_title('英伟达 近一年 收盘 + 20 日均线 / 成交量'); axp.set_ylabel('价格(美元)'); axp.legend()
axv.bar(one.index, one['Volume'] / 1e6, color='#b48ead', width=1.0)
axv.set_ylabel('成交量(百万股)')
plt.tight_layout(); plt.savefig('chart_03.png', dpi=120); plt.close()
print('图3(单只 K 线 + 成交量)已保存')

print(f'\n[小结] 五年最强 {final.index[0]} {final.iloc[0]:.1f} 倍,最弱 {final.index[-1]} {final.iloc[-1]:.1f} 倍')
print('\n[done] 中国版 3 张图已生成(yfinance + CSV 双路径),output.txt 见上方全部打印')

==== 1. 拉一篮子美股:CSV 优先,失败回退 yfinance(国内网络更稳)====
数据来源:本地缓存 /mnt/d/huizi_ai_project/ai_course_video/out/notebook/cn/../data/D043_yfinance_中国版.csv
各只股票缺失天数(NaN):
英伟达     0
特斯拉     0
微软      0
亚马逊     0
Meta    0
dtype: int64
对齐后剩 1256 个共同交易日  2021-06-14 → 2026-06-12

==== 2. 五年涨幅大比拼(净值=各自起点归 1)====
五年净值(起点 1,数字=翻了几倍):
  英伟达     11.43 倍
  特斯拉      1.97 倍
  Meta     1.70 倍
  微软       1.57 倍
  亚马逊      1.41 倍
图1(五年净值对比)已保存

==== 3. 复权到底复了什么:Close 已调拆股,Adj Close 再把分红也算进去 ====
数据来源:本地缓存 /mnt/d/huizi_ai_project/ai_course_video/out/notebook/cn/../data/D043_yfinance_中国版_msft.csv
图2(复权对比)已保存:起点 Adj Close 比 Close 低 4.2%,这就是分红被算进真实收益
Adj Close 才是含分红的真复权价;下一节 akshare 拉 A 股记得显式选复权(adjust='qfq')

==== 4. 单只 K 线 + 成交量(英伟达近一年)====
数据来源:本地缓存 /mnt/d/huizi_ai_project/ai_course_video/out/notebook/cn/../data/D043_yfinance_中国版_nvda1y.csv
图3(单只 K 线 + 成交量)已保存

[小结] 五年最强 英伟达 11.4 倍,最弱 亚马逊 1.4 倍

[done] 中国版 3 张图已生成(yfinance + CSV 双路径),output.txt 见上方全部打印


## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 美股 | NVDA/TSLA/MSFT/AMZN/META | 批量拉五只美股科技龙头5 年行情,净值归一比谁涨得最猛 |
| 复权 | MSFT | 微软 Close vs Adj Close:起点差约 4% 就是 5 年累计分红的贡献,Adj Close 才是含分红的真复权价 |
| 公司事件 | NVDA | 用 actions 查分红和拆股历史,理解原始价格为何跳变 |
| 跨市场拓展 | 0700.HK / BTC-USD | 同一套写法把代码换成港股腾讯 0700.HK 或加密 BTC-USD 就能拉,接口通用 |


## 常见坑

### ⚠ 01. 不复权算收益,全错

把 auto_adjust 设成 False 又直接拿 Close 去算长期收益,会漏掉分红(Close 没把分红算进去);在别的数据源用完全没调过的原始价,还会被拆股当天的跳变骗成假暴跌。算收益、做回测一律用复权价(auto_adjust=True,或取 Adj Close),这是最致命也最常见的错。

### ⚠ 02. 批量拉不限速被封

写个循环连续猛拉几十上百只,Yahoo 会把你当爬虫限流,轻则拿到空数据、重则临时封 IP。每只之间 time.sleep 歇一下,失败了重试,是基本礼貌也是稳定拉数据的前提。

### ⚠ 03. download 多股票列取错

yf.download 多只股票返回双层列(指标 × 代码),直接 df['Close'] 拿到的是所有股票的收盘,再用 columns= 强行改名极易错位(铁律 39 老坑)。要么像本节用 dict 构造,要么 raw['Close'][代码] 按名字取,绝不靠列顺序。

### ⚠ 04. 时区不统一对不齐

yfinance 返回的索引带美东时区,和国内数据或别的来源拼接时会因时区对不齐。统一用 index.tz_localize(None) 去掉时区再对齐、再画图,省得踩坑。

### ⚠ 05. 国内网络不稳直接卡死

国内访问 Yahoo 偶尔超时或被墙,代码会卡住或报错。正经做法是包一层 try/except,失败就回退到预先下载好的本地 CSV(本节中国版就是这么写的),别让联网失败把整个 notebook 卡死。

## 实战 SOP · yfinance 拉数据实战 6 条 SOP

1. 算收益一律用复权价 — 保持 auto_adjust=True,别用不复权 Close。
2. 批量拉必限速 — 每只 time.sleep 一下,失败重试,别被当爬虫。
3. 合并多股票用 dict 构造 — 列名对应数据,绝不用 columns= 按顺序赋值(铁律 39)。
4. 拉完先查缺失 — isna().sum() 看空洞,按市场对齐后再算。
5. 时区先归一 — tz_localize(None) 去时区,再对齐、再画图。
6. 国内联网包 try/except + CSV 回退 — 失败不卡死,读预下载的本地数据。

> 把这段打印贴在你电脑边。

## 总结 · 你应该带走的

2. yfinance 免费、免注册,一行 history(period='5y') 就拉到一只股票多年的开高低收成交量。
3. 批量拉一篮子用循环 + time.sleep 限速,合并表用 dict 构造防止列错位(铁律 39)。
4. 复权是命门:Close 默认已调拆股,Adj Close 再把分红算进去才是真复权价;算收益、做回测用复权价(auto_adjust=True)。
5. actions 查分红和拆股历史,帮你理解原始价格为什么会跳。
6. isna 校验缺失:上市晚、停牌会留空洞,跨市场要分市场对齐再算。
7. 国内网络不稳,中国版用 yfinance 优先 + 本地 CSV 回退,联网失败不卡死(铁律 55)。

## 自测题

**Q1.** 用 yfinance 拉一只股票5 年的日线,最简单的一行怎么写?(提示:yf.Ticker('NVDA').history(period='5y'),返回干净的单层列 Open/High/Low/Close/Volume;period 也可写 '1y'、'max',或用 start/end 指定起止。)

**Q2.** yfinance 里 Close 和 Adj Close 有什么区别,算收益用哪个?(提示:Close 默认已做拆股调整但没算分红;Adj Close 在它基础上把分红也调进去,是含分红的真复权价。设 auto_adjust=True 时 Close 就等于 Adj Close。算收益、做回测一律用复权价,否则会漏掉分红、或在别的数据源被拆股跳变骗。)

**Q3.** 批量拉一篮子股票要注意什么?怎么合并成一张表不出错?(提示:循环里每只 time.sleep 限速别被封;合并时用 dict 构造 DataFrame,列名就是股票名、和数据一一对应,绝不用 columns= 按顺序硬赋值,否则会列错位——铁律 39。)

**Q4.** 国内访问 Yahoo 不稳定,代码该怎么写才不会卡死?(提示:把拉数据包进 try/except,先尝试 yfinance,失败就回退去读预先下载好的本地 CSV;这就是铁律 55 的双路径写法,联网失败也能跑完。)

把答案写下来,3 天后再回看。

## 下一节预告

**Day 044 · akshare 拉 A 股** (akshare)

yfinance 拉海外方便,但 A 股的行情和财务它就不灵了。下一节请出国内数据的瑞士军刀 akshare:免费拉沪深 300 成分股、个股财报、宏观甚至另类数据,看看做 A 股研究怎么零成本把数据备齐。

## 推荐阅读

- yfinance 官方文档 / GitHub README(ranaroussi, 2024)— Ticker、history、download、actions 的权威用法
- Yahoo Finance《What is adjusted close?》(2024)— 复权价的定义与为什么算收益要用它
- Wes McKinney《Python for Data Analysis》(2022, O'Reilly)— 多张行情表的对齐、合并与缺失处理
- Yves Hilpisch《Python for Finance》(2018, O'Reilly)— 拿金融数据做收益与净值分析的完整流程
- Marcos López de Prado《Advances in Financial Machine Learning》(2018, Wiley)— 第一章谈数据质量,复权与对齐为何是回测的地基